Import required libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model

Data pre-processing & augmentation

In [ ]:
IMG_SIZE = (224, 224)
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    r"/content/drive/MyDrive/food-101/train",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    label_mode="categorical",
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    r"/content/drive/MyDrive/food-101/validation",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    label_mode="categorical",
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# create an object for data augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
    tf.keras.layers.RandomBrightness(0.1),
], name="data_augmentation")


Found 17100 files belonging to 18 classes.
Found 900 files belonging to 18 classes.


Model building

In [ ]:
base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(224,224,3),
    pooling='avg'
)

base_model.trainable = False   # Freeze for fast training

inputs = layers.Input(shape=(224,224,3))

# Apply augmentation
x = data_augmentation(inputs)

# EfficientNet preprocessing
x = tf.keras.applications.efficientnet.preprocess_input(x)

# Extract features
x = base_model(x)

# Custom classifier head
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(18, activation="softmax")(x)

model = Model(inputs, outputs)


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Compile optimizer & loss

In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 1280)           │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 18)             │         4,626 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,382,133 (16.72 MB)

 Trainable params: 332,562 (1.27 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

Train the model & validate

In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)


Epoch 1/20
535/535 ━━━━━━━━━━━━━━━━━━━━ 110s 175ms/step - accuracy: 0.6331 - loss: 1.2383 - val_accuracy: 0.8233 - val_loss: 0.6078
Epoch 2/20
535/535 ━━━━━━━━━━━━━━━━━━━━ 90s 168ms/step - accuracy: 0.7915 - loss: 0.6776 - val_accuracy: 0.8389 - val_loss: 0.5817
Epoch 3/20
535/535 ━━━━━━━━━━━━━━━━━━━━ 89s 167ms/step - accuracy: 0.8127 - loss: 0.5922 - val_accuracy: 0.8289 - val_loss: 0.5713
Epoch 4/20
535/535 ━━━━━━━━━━━━━━━━━━━━ 88s 165ms/step - accuracy: 0.8269 - loss: 0.5431 - val_accuracy: 0.8411 - val_loss: 0.5714
Epoch 5/20
535/535 ━━━━━━━━━━━━━━━━━━━━ 144s 168ms/step - accuracy: 0.8469 - loss: 0.4867 - val_accuracy: 0.8433 - val_loss: 0.5646
Epoch 6/20
535/535 ━━━━━━━━━━━━━━━━━━━━ 89s 166ms/step - accuracy: 0.8518 - loss: 0.4582 - val_accuracy: 0.8356 - val_loss: 0.5795
Epoch 7/20
535/535 ━━━━━━━━━━━━━━━━━━━━ 140s 163ms/step - accuracy: 0.8641 - loss: 0.4298 - val_accuracy: 0.8356 - val_loss: 0.5739
Epoch 8/20
535/535 ━━━━━━━━━━━━━━━━━━━━ 88s 165ms/step - accuracy: 0.8657 - loss

Save the model

In [11]:
model.save('food-20.keras')